In [1]:
!pip install google-colab-selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 117.0 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


# **SCRAPPING AVEC BOOKS.TOSCRAPE**

In [2]:
# importer packages
import pandas as pd
from selenium.webdriver.common.by import By
import google_colab_selenium as gs
import time

In [3]:
# Lancer le navigateur
driver = gs.Chrome()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
url = 'https://books.toscrape.com/catalogue/page-1.html'
# ouvrir la page
driver.get(url)


In [ ]:
#content
driver.page_source

In [5]:
containers = driver.find_elements(By.CSS_SELECTOR, 'article.product_pod' )

In [6]:
len(containers)

20

In [18]:
df_final = pd.DataFrame( )
for i in range(1, 51):
  url = f'https://books.toscrape.com/catalogue/page-{i}.html'
  # ouvrir la page
  driver.get(url)
  # containers
  containers = driver.find_elements(By.CSS_SELECTOR, 'article.product_pod')

  data = []
  for container in containers:
    try:

      dic = {
       'title': container.find_element(By.CSS_SELECTOR,'h3 a').get_attribute('title'),
       'price': container.find_element(By.CSS_SELECTOR,'p.price_color').text,
       'availability': container.find_element(By.CSS_SELECTOR, 'p.instock.availability').text.strip(),
       'star_rating': container.find_element(By.CSS_SELECTOR, 'p.star-rating').get_attribute('class').split(' ')[1],
       #'book_number': containers.index(container) + 1,
       'book_url': container.find_element(By.CSS_SELECTOR,'h3 a').get_attribute('href'),
       #'reviews' : container.find_element(By.XPATH,"//th[contains(text(), 'Number of reviews')]/following-sibling::td").text,
       #'product_description' : container.find_element(By.CSS_SELECTOR,'#product_description + p').text,
       #'product_type' : container.find_element(By.XPATH,"//th[contains(text(), 'Product Type')]/following-sibling::td").text,
       #'tax' : container.find_element(By.XPATH,"//th[contains(text(), 'Tax')]/following-sibling::td").text
          }

      data.append(dic)
    except:
      pass

  df = pd.DataFrame(data)
  df_final =pd.concat([df_final, df], axis = 0).reset_index(drop = True)

In [19]:
df_final

,title,price,availability,star_rating,book_url
0,A Light in the Attic,£51.77,In stock,Three,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,£53.74,In stock,One,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,£50.10,In stock,One,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,£47.82,In stock,Four,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,£54.23,In stock,Five,https://books.toscrape.com/catalogue/sapiens-a...
...,...,...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,£55.53,In stock,One,https://books.toscrape.com/catalogue/alice-in-...
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",£57.06,In stock,Four,https://books.toscrape.com/catalogue/ajin-demi...
997,A Spy's Devotion (The Regency Spies of London #1),£16.97,In stock,Five,https://books.toscrape.com/catalogue/a-spys-de...
998,1st to Die (Women's Murder Club #1),£53.98,In stock,One,https://books.toscrape.com/catalogue/1st-to-di...


In [20]:
# Création d'un secon dataframe pour récupérer les variables nécessitant l'ouverture de l'URL spécifique à chaque livre
databooks = []

for url in df_final['book_url']:

    driver.get(url)

    try:
        reviews = driver.find_element(By.XPATH,"//th[contains(text(), 'Number of reviews')]/following-sibling::td").text
    except:
        reviews = None

    try:product_description = driver.find_element(By.CSS_SELECTOR,'#product_description + p').text
    except:
        product_description = None

    try:
        product_type = driver.find_element(By.XPATH,"//th[contains(text(), 'Product Type')]/following-sibling::td").text
    except:
        product_type = None

    try:
        tax = driver.find_element(By.XPATH,"//th[contains(text(), 'Tax')]/following-sibling::td"
        ).text
    except:
        tax = None

    dic_details = {
        'book_url': url,
        'reviews': reviews,
        'product_description': product_description,
        'product_type': product_type,
        'tax': tax
    }

    databooks.append(dic_details)

dfbooks = pd.DataFrame(databooks)

In [22]:
dfbooks.tail()

,book_url,reviews,product_description,product_type,tax
995,https://books.toscrape.com/catalogue/alice-in-...,0,None,Books,£0.00
996,https://books.toscrape.com/catalogue/ajin-demi...,0,High school student Kei Nagai is struck dead i...,Books,£0.00
997,https://books.toscrape.com/catalogue/a-spys-de...,0,"In England’s Regency era, manners and elegance...",Books,£0.00
998,https://books.toscrape.com/catalogue/1st-to-di...,0,"James Patterson, bestselling author of the Ale...",Books,£0.00
999,https://books.toscrape.com/catalogue/1000-plac...,0,"Around the World, continent by continent, here...",Books,£0.00


In [23]:
#on fusionne les deux dtaframe
newdf = df_final.merge(dfbooks,on='book_url',how='left')

In [24]:
newdf.tail()

,title,price,availability,star_rating,book_url,reviews,product_description,product_type,tax
995,Alice in Wonderland (Alice's Adventures in Won...,£55.53,In stock,One,https://books.toscrape.com/catalogue/alice-in-...,0,None,Books,£0.00
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",£57.06,In stock,Four,https://books.toscrape.com/catalogue/ajin-demi...,0,High school student Kei Nagai is struck dead i...,Books,£0.00
997,A Spy's Devotion (The Regency Spies of London #1),£16.97,In stock,Five,https://books.toscrape.com/catalogue/a-spys-de...,0,"In England’s Regency era, manners and elegance...",Books,£0.00
998,1st to Die (Women's Murder Club #1),£53.98,In stock,One,https://books.toscrape.com/catalogue/1st-to-di...,0,"James Patterson, bestselling author of the Ale...",Books,£0.00
999,"1,000 Places to See Before You Die",£26.08,In stock,Five,https://books.toscrape.com/catalogue/1000-plac...,0,"Around the World, continent by continent, here...",Books,£0.00


# **SCRAPPING AVEC GAARAAS**

In [25]:
url = 'https://www.gaaraas.com/fr/users/dakar-auto?page=2'
# ouvrir la page
driver.get(url)

In [26]:
containersg = driver.find_elements(By.CSS_SELECTOR, 'a.common-ad-card')
len(containersg)

20

In [27]:
gcontainer = containersg[0]

In [29]:
df_finalg = pd.DataFrame( )
for i in range(1, 14):
  urlg = f'https://www.gaaraas.com/fr/users/dakar-auto?page-{i}'
  # ouvrir la page
  driver.get(urlg)
  # containers
  containersg = driver.find_elements(By.CSS_SELECTOR, 'a.common-ad-card')

  datag = []
  for container in containersg:
    try:

      dicg = {
       'marque': container.find_element(By.CSS_SELECTOR, 'div.specification-section').text.split(' ')[1],
       'modele': container.find_element(By.CSS_SELECTOR, 'div.specification-section').text.replace('\nDakar',' ').split(' ')[2],
       'annee': container.find_element(By.CSS_SELECTOR, 'div.specification-section').text.replace('\nDakar',' ').split(' ')[0],
       'prix': container.find_element(By.CSS_SELECTOR, 'span.price-wrap').text,
       'kilometrage': container.find_element(By.CSS_SELECTOR, 'div.ad-vehicle-mileage').text.replace('KILOMÉTRAGE\n', ' '),
       'transmission': container.find_element(By.CSS_SELECTOR, 'div.transmission').text,
       'region': container.find_element(By.CSS_SELECTOR, 'div.location').text
           }
      datag.append(dicg)
    except:
      pass

  dfg = pd.DataFrame(datag)
  df_finalg =pd.concat([df_finalg, dfg], axis = 0).reset_index(drop = True)

In [30]:
df_finalg.tail()

,marque,modele,annee,prix,kilometrage,transmission,region
255,Ford,Focus,2014,CFA 4 500 000,115 000 KM,Automatique,Dakar
256,Rover,600,2002,CFA 1 800 000,128 000 KM,Manuelle,Dakar
257,Volkswagen,Passat,2006,CFA 2 500 000,164 686 KM,Manuelle,Dakar
258,Peugeot,307,2007,CFA 2 500 000,160 000 KM,Manuelle,Dakar
259,Nissan,Murano,2008,CFA 3 200 000,115 305 KM,Automatique,Dakar
